In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve

# 1. Carregamento e Limpeza
def limpar_dados_e_cruzar():
    print("Carregando bases de dados...")
    df_2023 = pd.read_excel('BASE DE DADOS PEDE 2024 - DATATHON.xlsx', sheet_name='PEDE2023')
    df_2024 = pd.read_excel('BASE DE DADOS PEDE 2024 - DATATHON.xlsx', sheet_name='PEDE2024')

    def to_float(val):
        if pd.isna(val): return np.nan
        try: return float(str(val).replace(',', '.'))
        except: return np.nan

    def fix_idade(val):
        s = str(val).strip()
        if '1900-01-' in s:
            try: return float(s.split('-')[-1])
            except: return np.nan
        return to_float(val)

    def fix_fase(val):
        s = str(val).upper()
        if 'ALFA' in s: return 0
        if 'FASE' in s:
            try: return float(s.split('FASE')[-1].strip().split(' ')[0])
            except: pass
        return to_float(val)

    # 2023
    cols_23 = ['RA', 'Idade', 'IAA', 'IEG', 'IPS', 'IPP', 'IDA', 'IPV', 'IAN', 'Fase']
    df_23 = df_2023[cols_23].copy()
    df_23['Idade'] = df_23['Idade'].apply(fix_idade)
    df_23['Fase_Num'] = df_23['Fase'].apply(fix_fase)
    for c in ['IAA', 'IEG', 'IPS', 'IPP', 'IDA', 'IPV', 'IAN']:
        df_23[c] = df_23[c].apply(to_float)

    # 2024
    df_24 = df_2024[['RA', 'Defasagem']].copy()
    df_24['Defasagem_24'] = df_24['Defasagem'].apply(to_float)

    # Cruzamento Left Join (Mantém Evadidos)
    df_long = pd.merge(df_23, df_24[['RA', 'Defasagem_24']], on='RA', how='left')
    df_long['Target_Risco'] = ((df_long['Defasagem_24'] > 0) | (df_long['Defasagem_24'].isna())).astype(int)
    
    return df_long

# Execução
df_model_raw = limpar_dados_e_cruzar()

features = ['Idade', 'IAA', 'IEG', 'IPS', 'IPP', 'IDA', 'IPV', 'IAN', 'Fase_Num']
df_final = df_model_raw.dropna(subset=features).copy()

X = df_final[features]
y = df_final['Target_Risco']

# Divisão
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Treinamento "Tunado" (Mais profundo e agressivo)
print("Treinando modelo RandomForest Otimizado...")
modelo = RandomForestClassifier(
    n_estimators=400,             # Mais árvores
    max_depth=10,                 # Mais profundidade para pegar padrões complexos
    min_samples_leaf=2,           # Evita overfitting extremo
    class_weight='balanced_subsample', # Balanceamento avançado
    random_state=42
)
modelo.fit(X_train, y_train)

# 3. Otimização de Limiar (Foco em Encontrar os Alunos em Risco)
probs = modelo.predict_proba(X_test)[:, 1]

# Encontra o melhor ponto de corte para priorizar o Recall (Capturar a maioria das evasões)
precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

# Queremos pelo menos 75% de Recall (capturar 3 de cada 4 alunos em risco)
# Acha o limiar (threshold) que nos dá isso
target_recall = 0.75
idx = np.where(recalls >= target_recall)[0][-1] 
melhor_limiar = thresholds[idx] if idx < len(thresholds) else 0.5

print(f"\n--- Limiar de Decisão Otimizado: {melhor_limiar:.2f} ---")
preds_otimizadas = (probs >= melhor_limiar).astype(int)

print("\n--- Relatório de Classificação (Otimizado) ---")
print(classification_report(y_test, preds_otimizadas))
print(f"ROC-AUC Score Global: {roc_auc_score(y_test, probs):.4f}")

# Importância das Variáveis
imp = pd.DataFrame({'Atributo': features, 'Importancia': modelo.feature_importances_})
print("\n--- Ranking de Influência no Risco ---")
print(imp.sort_values(by='Importancia', ascending=False).to_string(index=False))

# Exportação
df_final['Probabilidade_Risco'] = modelo.predict_proba(X)[:, 1]
df_final.to_csv('predicao_risco_defasagem_evasao.csv', index=False)

Carregando bases de dados...
Treinando modelo RandomForest Otimizado...

--- Limiar de Decisão Otimizado: 0.36 ---

--- Relatório de Classificação (Otimizado) ---
              precision    recall  f1-score   support

           0       0.77      0.47      0.58        73
           1       0.43      0.75      0.55        40

    accuracy                           0.57       113
   macro avg       0.60      0.61      0.57       113
weighted avg       0.65      0.57      0.57       113

ROC-AUC Score Global: 0.6315

--- Ranking de Influência no Risco ---
Atributo  Importancia
     IPV     0.172781
     IDA     0.165461
     IEG     0.155021
     IPP     0.114155
     IAA     0.092500
     IPS     0.090991
Fase_Num     0.087744
   Idade     0.079964
     IAN     0.041385
